### Combine the data and prep it for training

In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
import datasets

DATA = "../data/"
MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

c:\Users\shubh\Desktop\Academics\Computational Creativity\Assignment 2\Bad_Poets_Society\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
gutenberg = pd.read_parquet(DATA + "gutenberg_clean.parquet")
pdp = pd.read_parquet(DATA + "pdp_clean.parquet")
print(len(gutenberg), "Gutenberg books")
print(len(pdp), "PDP poems")

869 Gutenberg books
34245 PDP poems


In [3]:
def clean_line(s):
    return re.sub(r"[^a-z]", "", s.lower())

gut_lines = set()
for text in gutenberg['text']:
    lines = text.split('\n')
    for line in lines:
        cl = clean_line(line)
        if len(cl) >= 20:
            gut_lines.add(cl)

def is_duplicate(text):
    lines = text.split('\n')
    clean_lines = [clean_line(l) for l in lines]
    valid_lines = [cl for cl in clean_lines if len(cl) >= 20]
    if not valid_lines:
        return False
    overlap = sum(1 for cl in valid_lines if cl in gut_lines)
    return overlap / len(valid_lines) > 0.5  

before = len(pdp)
pdp['is_dup'] = pdp['text'].apply(is_duplicate)
pdp = pdp[~pdp['is_dup']].drop(columns=['is_dup'])
print("Removed", before - len(pdp), "duplicate poems from PDP")

Removed 17060 duplicate poems from PDP


In [4]:
CANDIDATES = ["Emily Dickinson", "Christina Rossetti", "Edgar Allan Poe", "John Keats", "Walt Whitman", "John Milton"]

gut_cand = gutenberg[gutenberg['author'].isin(CANDIDATES)]
pdp_cand = pdp[pdp['author'].isin(CANDIDATES)]

all_texts = pd.concat([gut_cand[['author', 'text']], pdp_cand[['author', 'text']]])

chunks = []
for author, group in all_texts.groupby('author'):
    author_lines = []
    for text in group['text']:
        author_lines.extend(text.split('\n'))
    
    author_lines = [line for line in author_lines if line.strip()]

    for i in range(0, len(author_lines), 40):
        chunk_lines = author_lines[i:i+40]
        if len(chunk_lines) == 40:
            chunks.append({
                'author': author,
                'text': '\n'.join(chunk_lines)
            })

chunks_df = pd.DataFrame(chunks)
print(f"Created {len(chunks_df)} chunks of 40 lines for {len(chunks_df['author'].unique())} authors.")

Created 1855 chunks of 40 lines for 6 authors.


In [5]:
train_df, temp_df = train_test_split(chunks_df, test_size=0.2, random_state=42, stratify=chunks_df['author'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['author'])

print(f"Train   : {len(train_df)} chunks")
print(f"Val     : {len(val_df)} chunks")
print(f"Test    : {len(test_df)} chunks")

Train   : 1484 chunks
Val     : 185 chunks
Test    : 186 chunks


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=1024)

ds = datasets.DatasetDict({
    'train': datasets.Dataset.from_pandas(train_df, preserve_index=False),
    'val': datasets.Dataset.from_pandas(val_df, preserve_index=False),
    'test': datasets.Dataset.from_pandas(test_df, preserve_index=False)
})

tokenized_ds = ds.map(tokenize_function, batched=True)

tokenized_ds['train'].to_parquet(DATA + 'train_test/' + 'train.parquet')
tokenized_ds['val'].to_parquet(DATA   + 'train_test/' + 'val.parquet')
tokenized_ds['test'].to_parquet(DATA  + 'train_test/' + 'test.parquet')
print("Saved train, val, and test datasets to data/")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 62.91ba/s]

Saved train, val, and test datasets to data/
